In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

Libraries imported successfully!


In [ ]:
# Load the dataset
df = pd.read_csv('Crop_recommendation_with_season.csv')


print("\nFirst few rows:")
print(df.head())

print("\nDataset shape:", df.shape)
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names and data types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nBasic statistics:")
print(df.describe())

DATASET EXPLORATION

First few rows:
    N   P   K  temperature   humidity        ph    rainfall label  Season
0  90  42  43    20.879744  82.002744  6.502985  202.935536  rice  Kharif
1  85  58  41    21.770462  80.319644  7.038096  226.655537  rice  Kharif
2  60  55  44    23.004459  82.320763  7.840207  263.964248  rice  Kharif
3  74  35  40    26.491096  80.158363  6.980401  242.864034  rice  Kharif
4  78  42  42    20.130175  81.604873  7.628473  262.717340  rice  Kharif

Dataset shape: (2200, 9)
Number of rows: 2200
Number of columns: 9

Column names and data types:
N                int64
P                int64
K                int64
temperature    float64
humidity       float64
ph             float64
rainfall       float64
label              str
Season             str
dtype: object

Missing values:
N              0
P              0
K              0
temperature    0
humidity       0
ph             0
rainfall       0
label          0
Season         0
dtype: int64

Basic statistics

In [ ]:
# Data preprocessing and cleaning
print("DATA PREPROCESSING")


# Check for duplicate rows
duplicate_rows = df.duplicated().sum()
print(f"\nDuplicate rows found: {duplicate_rows}")

if duplicate_rows > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Duplicates removed. New dataset shape: {df.shape}")

# Check target column
print(f"\nTarget column (label) unique values: {df['label'].nunique()}")
print(f"\nClass distribution:")
print(df['label'].value_counts())
print(f"\nClass distribution (%):")
print(df['label'].value_counts(normalize=True) * 100)


DATA PREPROCESSING

Duplicate rows found: 0

Target column (label) unique values: 22

Class distribution:
label
rice           100
maize          100
chickpea       100
kidneybeans    100
pigeonpeas     100
mothbeans      100
mungbean       100
blackgram      100
lentil         100
pomegranate    100
banana         100
mango          100
grapes         100
watermelon     100
muskmelon      100
apple          100
orange         100
papaya         100
coconut        100
cotton         100
jute           100
coffee         100
Name: count, dtype: int64

Class distribution (%):
label
rice           4.545455
maize          4.545455
chickpea       4.545455
kidneybeans    4.545455
pigeonpeas     4.545455
mothbeans      4.545455
mungbean       4.545455
blackgram      4.545455
lentil         4.545455
pomegranate    4.545455
banana         4.545455
mango          4.545455
grapes         4.545455
watermelon     4.545455
muskmelon      4.545455
apple          4.545455
orange         4.545455
papa

In [ ]:
# Separate features and target
print("FEATURE-TARGET SEPARATION")


# Separate X (features) and y (target)
X = df.drop('label', axis=1)
y = df['label']

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")

print(f"\nFeature columns: {list(X.columns)}")

# Identify categorical columns (for encoding)
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
print(f"\nCategorical columns: {categorical_cols}")

# Encode categorical variables using LabelEncoder
# IMPORTANT: Create encoders for each categorical column to avoid data leakage
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le
    print(f"Encoded column '{col}': {dict(zip(le.classes_, le.transform(le.classes_)))}")

print(f"\nFeatures after encoding shape: {X.shape}")
print("\nFeatures (first few rows after encoding):")
print(X.head())


FEATURE-TARGET SEPARATION

Features shape: (2200, 8)
Target shape: (2200,)

Feature columns: ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'Season']

Categorical columns: ['Season']
Encoded column 'Season': {'Kharif': np.int64(0), 'Rabi': np.int64(1), 'Transition': np.int64(2)}

Features after encoding shape: (2200, 8)

Features (first few rows after encoding):
    N   P   K  temperature   humidity        ph    rainfall  Season
0  90  42  43    20.879744  82.002744  6.502985  202.935536       0
1  85  58  41    21.770462  80.319644  7.038096  226.655537       0
2  60  55  44    23.004459  82.320763  7.840207  263.964248       0
3  74  35  40    26.491096  80.158363  6.980401  242.864034       0
4  78  42  42    20.130175  81.604873  7.628473  262.717340       0


C:\Users\Rishabh\AppData\Local\Temp\ipykernel_17416\1162070386.py:16: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=['object']).columns.tolist()


In [ ]:
# Train-test split with stratification

print("TRAIN-TEST SPLIT")

# Split with stratification to maintain class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"\nTraining set size: {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Testing set size: {X_test.shape[0]} ({X_test.shape[0]/len(X)*100:.1f}%)")

print(f"\nTraining set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

print("\nClass distribution in training set:")
print(y_train.value_counts())

print("\nClass distribution in testing set:")
print(y_test.value_counts())

print("\n✓ No data leakage: Target column excluded from features")
print("✓ Stratified split maintains class distribution")


TRAIN-TEST SPLIT

Training set size: 1760 (80.0%)
Testing set size: 440 (20.0%)

Training set shape: (1760, 8)
Testing set shape: (440, 8)

Class distribution in training set:
label
orange         80
grapes         80
kidneybeans    80
mothbeans      80
cotton         80
banana         80
lentil         80
mungbean       80
coffee         80
muskmelon      80
apple          80
blackgram      80
pigeonpeas     80
maize          80
rice           80
watermelon     80
jute           80
mango          80
pomegranate    80
papaya         80
coconut        80
chickpea       80
Name: count, dtype: int64

Class distribution in testing set:
label
orange         20
banana         20
cotton         20
maize          20
chickpea       20
rice           20
blackgram      20
watermelon     20
pomegranate    20
mothbeans      20
grapes         20
mango          20
apple          20
kidneybeans    20
jute           20
coffee         20
coconut        20
papaya         20
lentil         20
mungbean   

In [ ]:
# Train multiple models for comparison

print("MODEL TRAINING - BASELINE & IMPROVED MODELS")


# ============================================================
# 1. Decision Tree Classifier (with controlled depth to reduce overfitting)
# ============================================================
dt_classifier = DecisionTreeClassifier(max_depth=10, random_state=42, min_samples_split=5, min_samples_leaf=2)
dt_classifier.fit(X_train, y_train)

print("\n✓ DecisionTreeClassifier (controlled depth=10) trained successfully!")
print(f"  Key parameters: max_depth={dt_classifier.max_depth}, "
      f"min_samples_split={dt_classifier.min_samples_split}, "
      f"min_samples_leaf={dt_classifier.min_samples_leaf}")

# Make predictions for Decision Tree
y_train_pred_dt = dt_classifier.predict(X_train)
y_test_pred_dt = dt_classifier.predict(X_test)

# ============================================================
# 2. Random Forest Classifier (stronger ensemble model)
# ============================================================
rf_classifier = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, 
                                       min_samples_split=5, min_samples_leaf=2, n_jobs=-1)
rf_classifier.fit(X_train, y_train)

print("\n✓ RandomForestClassifier (100 trees, depth=10) trained successfully!")
print(f"  Key parameters: n_estimators={rf_classifier.n_estimators}, "
      f"max_depth={rf_classifier.max_depth}, "
      f"min_samples_split={rf_classifier.min_samples_split}")

# Make predictions for Random Forest
y_train_pred_rf = rf_classifier.predict(X_train)
y_test_pred_rf = rf_classifier.predict(X_test)

print(f"\n✓ All models trained and predictions completed")


MODEL TRAINING - BASELINE & IMPROVED MODELS

✓ DecisionTreeClassifier (controlled depth=10) trained successfully!
  Key parameters: max_depth=10, min_samples_split=5, min_samples_leaf=2

✓ RandomForestClassifier (100 trees, depth=10) trained successfully!
  Key parameters: n_estimators=100, max_depth=10, min_samples_split=5

✓ All models trained and predictions completed


In [ ]:
# Model evaluation and comparison

print("MODEL EVALUATION AND COMPARISON")


# ============================================================
# Calculate accuracy scores for both models
# ============================================================
train_accuracy_dt = accuracy_score(y_train, y_train_pred_dt)
test_accuracy_dt = accuracy_score(y_test, y_test_pred_dt)

train_accuracy_rf = accuracy_score(y_train, y_train_pred_rf)
test_accuracy_rf = accuracy_score(y_test, y_test_pred_rf)

# ============================================================
# Display accuracy comparison
# ============================================================
print(f"\n{'='*60}")
print(f"MODEL ACCURACY COMPARISON")
print(f"{'='*60}")

print(f"\n{'Model':<25} {'Train Accuracy':<20} {'Test Accuracy':<20}")
print("-" * 65)
print(f"{'Decision Tree (DT)':<25} {train_accuracy_dt:.4f} ({train_accuracy_dt*100:6.2f}%) {'':>5} {test_accuracy_dt:.4f} ({test_accuracy_dt*100:6.2f}%)")
print(f"{'Random Forest (RF)':<25} {train_accuracy_rf:.4f} ({train_accuracy_rf*100:6.2f}%) {'':>5} {test_accuracy_rf:.4f} ({test_accuracy_rf*100:6.2f}%)")

# ============================================================
# Detailed Classification Report for both models
# ============================================================
print(f"\n{'='*60}")
print("DECISION TREE - CLASSIFICATION REPORT (Test Set)")
print(f"{'='*60}")
print(classification_report(y_test, y_test_pred_dt))

print(f"\n{'='*60}")
print("RANDOM FOREST - CLASSIFICATION REPORT (Test Set)")
print(f"{'='*60}")
print(classification_report(y_test, y_test_pred_rf))

# ============================================================
# Confusion Matrices
# ============================================================
print(f"\n{'='*60}")
print("DECISION TREE - CONFUSION MATRIX (Test Set)")
print(f"{'='*60}")
cm_dt = confusion_matrix(y_test, y_test_pred_dt)
classes = sorted(y.unique())
print("Note: Showing first 5x5 rows/cols for readability")
print(cm_dt[:5, :5])

print(f"\n{'='*60}")
print("RANDOM FOREST - CONFUSION MATRIX (Test Set)")
print(f"{'='*60}")
cm_rf = confusion_matrix(y_test, y_test_pred_rf)
print("Note: Showing first 5x5 rows/cols for readability")
print(cm_rf[:5, :5])


MODEL EVALUATION AND COMPARISON

MODEL ACCURACY COMPARISON

Model                     Train Accuracy       Test Accuracy       
-----------------------------------------------------------------
Decision Tree (DT)        0.9812 ( 98.12%)       0.9591 ( 95.91%)
Random Forest (RF)        0.9977 ( 99.77%)       0.9932 ( 99.32%)

DECISION TREE - CLASSIFICATION REPORT (Test Set)
              precision    recall  f1-score   support

       apple       1.00      1.00      1.00        20
      banana       1.00      1.00      1.00        20
   blackgram       1.00      0.80      0.89        20
    chickpea       1.00      1.00      1.00        20
     coconut       1.00      1.00      1.00        20
      coffee       1.00      1.00      1.00        20
      cotton       1.00      1.00      1.00        20
      grapes       1.00      1.00      1.00        20
        jute       0.65      1.00      0.78        20
 kidneybeans       1.00      1.00      1.00        20
      lentil       0.86     

In [12]:
# 5-Fold Cross-Validation Analysis
print("\n" + "="*60)
print("5-FOLD CROSS-VALIDATION ANALYSIS")
print("="*60)

# ============================================================
# Cross-validation for Decision Tree
# ============================================================
cv_scores_dt = cross_val_score(dt_classifier, X_train, y_train, cv=5, scoring='accuracy')
mean_cv_dt = cv_scores_dt.mean()
std_cv_dt = cv_scores_dt.std()

print(f"\nDecision Tree - Cross-Validation Scores (5-fold):")
for i, score in enumerate(cv_scores_dt, 1):
    print(f"  Fold {i}: {score:.4f}")
print(f"  Mean CV Score:     {mean_cv_dt:.4f} (+/- {std_cv_dt:.4f})")

# ============================================================
# Cross-validation for Random Forest
# ============================================================
cv_scores_rf = cross_val_score(rf_classifier, X_train, y_train, cv=5, scoring='accuracy')
mean_cv_rf = cv_scores_rf.mean()
std_cv_rf = cv_scores_rf.std()

print(f"\nRandom Forest - Cross-Validation Scores (5-fold):")
for i, score in enumerate(cv_scores_rf, 1):
    print(f"  Fold {i}: {score:.4f}")
print(f"  Mean CV Score:     {mean_cv_rf:.4f} (+/- {std_cv_rf:.4f})")

# ============================================================
# Generalization Performance Summary
# ============================================================
print(f"\n{'='*60}")
print("GENERALIZATION PERFORMANCE SUMMARY")
print(f"{'='*60}")

print(f"\n{'Model':<25} {'Train Acc':<15} {'Test Acc':<15} {'CV Mean':<15} {'CV Std':<15}")
print("-" * 85)
print(f"{'Decision Tree':<25} {train_accuracy_dt:.4f} {test_accuracy_dt:.4f} {mean_cv_dt:.4f} {std_cv_dt:.4f}")
print(f"{'Random Forest':<25} {train_accuracy_rf:.4f} {test_accuracy_rf:.4f} {mean_cv_rf:.4f} {std_cv_rf:.4f}")

# ============================================================
# Overfitting Analysis
# ============================================================
print(f"\n{'='*60}")
print("OVERFITTING ANALYSIS")
print(f"{'='*60}")

overfitting_gap_dt = train_accuracy_dt - test_accuracy_dt
overfitting_gap_rf = train_accuracy_rf - test_accuracy_rf

print(f"\nDecision Tree:")
print(f"  Training-Testing Gap: {overfitting_gap_dt:.4f} ({overfitting_gap_dt*100:.2f}%)")
print(f"  Status: {'✓ Good generalization' if overfitting_gap_dt < 0.05 else '✗ Potential overfitting'}")

print(f"\nRandom Forest:")
print(f"  Training-Testing Gap: {overfitting_gap_rf:.4f} ({overfitting_gap_rf*100:.2f}%)")
print(f"  Status: {'✓ Good generalization' if overfitting_gap_rf < 0.05 else '✗ Potential overfitting'}")


5-FOLD CROSS-VALIDATION ANALYSIS

Decision Tree - Cross-Validation Scores (5-fold):
  Fold 1: 0.9688
  Fold 2: 0.9773
  Fold 3: 0.9886
  Fold 4: 0.9773
  Fold 5: 0.9688
  Mean CV Score:     0.9761 (+/- 0.0073)

Random Forest - Cross-Validation Scores (5-fold):
  Fold 1: 0.9915
  Fold 2: 0.9915
  Fold 3: 1.0000
  Fold 4: 0.9972
  Fold 5: 0.9830
  Mean CV Score:     0.9926 (+/- 0.0058)

GENERALIZATION PERFORMANCE SUMMARY

Model                     Train Acc       Test Acc        CV Mean         CV Std         
-------------------------------------------------------------------------------------
Decision Tree             0.9812 0.9591 0.9761 0.0073
Random Forest             0.9977 0.9932 0.9926 0.0058

OVERFITTING ANALYSIS

Decision Tree:
  Training-Testing Gap: 0.0222 (2.22%)
  Status: ✓ Good generalization

Random Forest:
  Training-Testing Gap: 0.0045 (0.45%)
  Status: ✓ Good generalization


In [ ]:

# ============================================================
# Random Forest Feature Importance
# ============================================================
feature_importance_rf = rf_classifier.feature_importances_
feature_names = X.columns

# Create a dataframe for better visualization
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance_rf
}).sort_values('Importance', ascending=False)

print("\nRandom Forest - Feature Importance Ranking:")
print("="*60)
for idx, row in importance_df.iterrows():
    bar_length = int(row['Importance'] * 50)
    print(f"{row['Feature']:<15} {row['Importance']:.6f} {'█' * bar_length}")

# ============================================================
# Decision Tree Feature Importance
# ============================================================
feature_importance_dt = dt_classifier.feature_importances_

importance_dt_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance_dt
}).sort_values('Importance', ascending=False)

print("\n\nDecision Tree - Feature Importance Ranking:")
print("="*60)
for idx, row in importance_dt_df.iterrows():
    bar_length = int(row['Importance'] * 50)
    print(f"{row['Feature']:<15} {row['Importance']:.6f} {'█' * bar_length}")

# ============================================================
# Feature Importance Comparison Visualization
# ============================================================
print("\n\n" + "="*60)
print("FEATURE IMPORTANCE SUMMARY")
print("="*60)

print("\nTop 5 Features by Random Forest:")
print(importance_df.head().to_string(index=False))

print("\n\nTop 5 Features by Decision Tree:")
print(importance_dt_df.head().to_string(index=False))

print("\n\n" + "="*60)
print("KEY INSIGHTS")
print("="*60)
print(f"\nMost important feature (Random Forest): {importance_df.iloc[0]['Feature']} ({importance_df.iloc[0]['Importance']:.4f})")
print(f"Most important feature (Decision Tree): {importance_dt_df.iloc[0]['Feature']} ({importance_dt_df.iloc[0]['Importance']:.4f})")


FEATURE IMPORTANCE ANALYSIS

Random Forest - Feature Importance Ranking:
humidity        0.216948 ██████████
rainfall        0.196557 █████████
K               0.177643 ████████
P               0.141585 ███████
N               0.100661 █████
temperature     0.074825 ███
ph              0.053426 ██
Season          0.038355 █


Decision Tree - Feature Importance Ranking:
rainfall        0.314602 ███████████████
P               0.230441 ███████████
N               0.196632 █████████
humidity        0.154202 ███████
K               0.098263 ████
temperature     0.003463 
Season          0.002397 
ph              0.000000 


FEATURE IMPORTANCE SUMMARY

Top 5 Features by Random Forest:
 Feature  Importance
humidity    0.216948
rainfall    0.196557
       K    0.177643
       P    0.141585
       N    0.100661


Top 5 Features by Decision Tree:
 Feature  Importance
rainfall    0.314602
       P    0.230441
       N    0.196632
humidity    0.154202
       K    0.098263


KEY INSIGHTS

Most im

In [14]:
# Final Summary and Recommendations
print("\n" + "="*60)
print("FINAL SUMMARY & RECOMMENDATIONS")
print("="*60)

print(f"\n{'='*60}")
print("MODEL PERFORMANCE COMPARISON")
print(f"{'='*60}")

comparison_data = {
    'Metric': ['Train Accuracy', 'Test Accuracy', 'CV Mean Score', 'CV Std Dev', 'Overfitting Gap'],
    'Decision Tree': [f"{train_accuracy_dt:.4f}", f"{test_accuracy_dt:.4f}", 
                     f"{mean_cv_dt:.4f}", f"{std_cv_dt:.4f}", f"{overfitting_gap_dt:.4f}"],
    'Random Forest': [f"{train_accuracy_rf:.4f}", f"{test_accuracy_rf:.4f}", 
                     f"{mean_cv_rf:.4f}", f"{std_cv_rf:.4f}", f"{overfitting_gap_rf:.4f}"]
}

comparison_df = pd.DataFrame(comparison_data)
print("\n" + comparison_df.to_string(index=False))

print(f"\n{'='*60}")
print("RECOMMENDATIONS")
print(f"{'='*60}")

if test_accuracy_rf > test_accuracy_dt:
    print(f"\n✓ RECOMMENDED MODEL: Random Forest")
    print(f"  Reason: Higher test accuracy ({test_accuracy_rf:.4f} vs {test_accuracy_dt:.4f})")
    print(f"  CV robustness: {mean_cv_rf:.4f} vs {mean_cv_dt:.4f}")
else:
    print(f"\n✓ RECOMMENDED MODEL: Decision Tree")
    print(f"  Reason: Higher test accuracy ({test_accuracy_dt:.4f} vs {test_accuracy_rf:.4f})")

if overfitting_gap_rf < overfitting_gap_dt:
    print(f"\n✓ Better generalization: Random Forest (gap: {overfitting_gap_rf:.4f} vs {overfitting_gap_dt:.4f})")
else:
    print(f"\n✓ Better generalization: Decision Tree (gap: {overfitting_gap_dt:.4f} vs {overfitting_gap_rf:.4f})")

print(f"\n✓ Cross-validation stability: ", end="")
if std_cv_rf < std_cv_dt:
    print(f"Random Forest (std: {std_cv_rf:.4f} vs {std_cv_dt:.4f})")
else:
    print(f"Decision Tree (std: {std_cv_dt:.4f} vs {std_cv_rf:.4f})")

print(f"\n{'='*60}")
print("PIPELINE EXECUTION COMPLETED SUCCESSFULLY!")
print(f"{'='*60}")
print("\nDataset: Crop_recommendation_with_season.csv")
print(f"Total Samples: {len(df)}")
print(f"Training Samples: {len(X_train)}")
print(f"Testing Samples: {len(X_test)}")
print(f"Number of Classes: {df['label'].nunique()}")
print(f"Number of Features: {X.shape[1]}")
print(f"\nNo data leakage verified: ✓")
print(f"Stratified split maintained: ✓")
print(f"Models evaluated comprehensively: ✓")


FINAL SUMMARY & RECOMMENDATIONS

MODEL PERFORMANCE COMPARISON

         Metric Decision Tree Random Forest
 Train Accuracy        0.9812        0.9977
  Test Accuracy        0.9591        0.9932
  CV Mean Score        0.9761        0.9926
     CV Std Dev        0.0073        0.0058
Overfitting Gap        0.0222        0.0045

RECOMMENDATIONS

✓ RECOMMENDED MODEL: Random Forest
  Reason: Higher test accuracy (0.9932 vs 0.9591)
  CV robustness: 0.9926 vs 0.9761

✓ Better generalization: Random Forest (gap: 0.0045 vs 0.0222)

✓ Cross-validation stability: Random Forest (std: 0.0058 vs 0.0073)

PIPELINE EXECUTION COMPLETED SUCCESSFULLY!

Dataset: Crop_recommendation_with_season.csv
Total Samples: 2200
Training Samples: 1760
Testing Samples: 440
Number of Classes: 22
Number of Features: 8

No data leakage verified: ✓
Stratified split maintained: ✓
Models evaluated comprehensively: ✓


In [16]:
# Save the Random Forest model using joblib
print("\n" + "="*60)
print("MODEL PERSISTENCE")
print("="*60)

# Save the trained Random Forest model
model_filename = 'random_forest_crop_model.pkl'
joblib.dump(rf_classifier, model_filename)

print(f"\n✓ Random Forest model saved successfully!")
print(f"  File: {model_filename}")
print(f"  Model type: {type(rf_classifier)}")
print(f"  Model parameters: n_estimators={rf_classifier.n_estimators}, max_depth={rf_classifier.max_depth}")

# Save the label encoders for preprocessing new data
encoders_filename = 'label_encoders.pkl'
joblib.dump(label_encoders, encoders_filename)

print(f"\n✓ Label encoders saved successfully!")
print(f"  File: {encoders_filename}")
print(f"  Encoders: {list(label_encoders.keys())}")

# Save feature names for reference
feature_names_filename = 'feature_names.pkl'
joblib.dump(feature_names, feature_names_filename)

print(f"\n✓ Feature names saved successfully!")
print(f"  File: {feature_names_filename}")
print(f"  Features: {list(feature_names)}")

print(f"\n{'='*60}")
print("Model and preprocessing artifacts ready for deployment!")
print(f"{'='*60}")

# Example code for loading the model (for reference)
print("\n" + "="*60)
print("HOW TO LOAD THE MODEL")
print("="*60)
print("""
# Load the saved model
loaded_model = joblib.load('random_forest_crop_model.pkl')

# Load the encoders
loaded_encoders = joblib.load('label_encoders.pkl')

# Load feature names
loaded_features = joblib.load('feature_names.pkl')

# Use the loaded model for predictions on new data
# predictions = loaded_model.predict(new_data)
""")


MODEL PERSISTENCE

✓ Random Forest model saved successfully!
  File: random_forest_crop_model.pkl
  Model type: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
  Model parameters: n_estimators=100, max_depth=10

✓ Label encoders saved successfully!
  File: label_encoders.pkl
  Encoders: ['Season']

✓ Feature names saved successfully!
  File: feature_names.pkl
  Features: ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'Season']

Model and preprocessing artifacts ready for deployment!

HOW TO LOAD THE MODEL

# Load the saved model
loaded_model = joblib.load('random_forest_crop_model.pkl')

# Load the encoders
loaded_encoders = joblib.load('label_encoders.pkl')

# Load feature names
loaded_features = joblib.load('feature_names.pkl')

# Use the loaded model for predictions on new data
# predictions = loaded_model.predict(new_data)

